# Dancing Links (DLX)

Dancing Links (Knuth, 2000) is a technique for efficiently implementing **Algorithm X**, a backtracking search that finds all solutions to the **exact cover problem**. The core data structure is a sparse binary matrix represented as a grid of **circular doubly-linked nodes** — each node links to its neighbors in four directions (up, down, left, right) plus a reference to its column header.

The key insight is that removing a node from a doubly-linked list and later restoring it are both **O(1)** operations:

- **Unlink:** `node.right.left = node.left; node.left.right = node.right`
- **Relink:** `node.right.left = node; node.left.right = node`

Because the node itself is never destroyed, it still remembers where it was — so relinking is trivial. Knuth calls this the "dance" of the pointers: nodes leave the structure and return to it as the algorithm backtracks, and the pointers shuffle back and forth like partners in a dance.

This makes DLX ideal for constraint satisfaction problems that can be expressed as exact cover: **Sudoku**, **N-Queens**, **pentomino tiling**, and many others.

## Implementation

The sparse matrix is represented as a grid of `Node` objects linked in four directions. Each column has a `ColumnHeader` that tracks how many nodes remain in that column (`size`). A master header node links all column headers into a horizontal list.

Algorithm X proceeds recursively:
1. Choose the column with the fewest 1s (the **S-heuristic** — minimizes branching).
2. For each row in that column, tentatively include it in the solution and **cover** every column it touches.
3. Recurse. If all columns are covered, a solution is found.
4. **Uncover** columns in reverse order (the pointers dance back) and try the next row.

In [5]:
class Node:
    """A node in the doubly-linked sparse matrix."""
    __slots__ = ('left', 'right', 'up', 'down', 'column', 'row')

    def __init__(self, column=None, row=None):
        self.left = self
        self.right = self
        self.up = self
        self.down = self
        self.column = column
        self.row = row


class ColumnHeader(Node):
    """Column header node — tracks column size and name."""
    __slots__ = ('size', 'name')

    def __init__(self, name):
        super().__init__()
        self.column = self
        self.size = 0
        self.name = name
        self.row = -1

    def __repr__(self):
        return f"Column({self.name}, size={self.size})"


class DLX:
    """Dancing Links implementation of Algorithm X for exact cover."""

    def __init__(self, matrix, column_names=None):
        self.header = Node()  # master header
        self.solution = []
        self.columns = []
        self._build(matrix, column_names)

    def _build(self, matrix, column_names):
        num_cols = len(matrix[0]) if matrix else 0
        if column_names is None:
            column_names = list(range(num_cols))

        # Create column headers linked horizontally.
        prev = self.header
        for name in column_names:
            col = ColumnHeader(name)
            self.columns.append(col)
            col.left = prev
            col.right = self.header
            prev.right = col
            self.header.left = col
            prev = col

        # Create nodes for each 1 in the matrix.
        for r, row in enumerate(matrix):
            first = None
            for c, val in enumerate(row):
                if val:
                    col = self.columns[c]
                    node = Node(column=col, row=r)
                    # Link vertically — insert at the bottom of the column.
                    node.down = col
                    node.up = col.up
                    col.up.down = node
                    col.up = node
                    col.size += 1
                    # Link horizontally within the row.
                    if first is None:
                        first = node
                        node.left = node
                        node.right = node
                    else:
                        node.left = first.left
                        node.right = first
                        first.left.right = node
                        first.left = node

    def _cover(self, col):
        """Remove a column and all rows that intersect it."""
        col.right.left = col.left
        col.left.right = col.right
        i = col.down
        while i is not col:
            j = i.right
            while j is not i:
                j.down.up = j.up
                j.up.down = j.down
                j.column.size -= 1
                j = j.right
            i = i.down

    def _uncover(self, col):
        """Restore a column and all rows that intersect it (reverse of cover)."""
        i = col.up
        while i is not col:
            j = i.left
            while j is not i:
                j.column.size += 1
                j.down.up = j
                j.up.down = j
                j = j.left
            i = i.up
        col.right.left = col
        col.left.right = col

    def _choose_column(self):
        """S-heuristic: choose the column with the smallest size."""
        min_size = float('inf')
        best = None
        j = self.header.right
        while j is not self.header:
            if j.size < min_size:
                min_size = j.size
                best = j
            j = j.right
        return best

    def solve(self):
        """Yield all solutions. Each solution is a list of row indices."""
        yield from self._search(0)

    def _search(self, depth):
        if self.header.right is self.header:
            yield list(self.solution)
            return

        col = self._choose_column()
        if col.size == 0:
            return

        self._cover(col)
        r = col.down
        while r is not col:
            self.solution.append(r.row)
            j = r.right
            while j is not r:
                self._cover(j.column)
                j = j.right

            yield from self._search(depth + 1)

            self.solution.pop()
            j = r.left
            while j is not r:
                self._uncover(j.column)
                j = j.left

            r = r.down
        self._uncover(col)

## Example: Step by Step

Let's trace the algorithm on Knuth's classic example matrix. The goal is to find a subset of rows that covers every column exactly once.

In [6]:
# Knuth's example matrix from the Dancing Links paper.
matrix = [
    [1, 0, 0, 1, 0, 0, 1],
    [1, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 1],
    [0, 0, 1, 0, 1, 1, 0],
    [0, 1, 1, 0, 0, 1, 1],
    [0, 1, 0, 0, 0, 0, 1],
]
col_names = list("ABCDEFG")

print("Matrix:")
print("     " + "  ".join(col_names))
for i, row in enumerate(matrix):
    print(f"  {i}: {row}")


# A variant of DLX.solve that prints what it's doing.
def solve_with_trace(dlx):
    yield from _search_trace(dlx, 0)

def _search_trace(dlx, depth):
    indent = "  " * depth
    if dlx.header.right is dlx.header:
        print(f"{indent}** All columns covered — solution found: rows {dlx.solution}")
        yield list(dlx.solution)
        return
    col = dlx._choose_column()
    print(f"{indent}Choose column {col.name} (size={col.size})")
    if col.size == 0:
        print(f"{indent}Dead end — column {col.name} has no rows, backtracking")
        return
    dlx._cover(col)
    r = col.down
    while r is not col:
        dlx.solution.append(r.row)
        covered_cols = []
        j = r.right
        while j is not r:
            covered_cols.append(str(j.column.name))
            dlx._cover(j.column)
            j = j.right
        extra = f", also covering [{', '.join(covered_cols)}]" if covered_cols else ""
        print(f"{indent}Try row {r.row}{extra}")
        yield from _search_trace(dlx, depth + 1)
        dlx.solution.pop()
        j = r.left
        while j is not r:
            dlx._uncover(j.column)
            j = j.left
        r = r.down
    dlx._uncover(col)
    print(f"{indent}Uncovered column {col.name}, backtracking")


print("\nSolving with trace:")
dlx = DLX(matrix, col_names)
solutions = list(solve_with_trace(dlx))

print(f"\nSolution rows: {solutions[0]}")
print("Verification:")
for r in solutions[0]:
    cols = [col_names[c] for c, v in enumerate(matrix[r]) if v]
    print(f"  Row {r}: covers columns {cols}")

Matrix:
     A  B  C  D  E  F  G
  0: [1, 0, 0, 1, 0, 0, 1]
  1: [1, 0, 0, 1, 0, 0, 0]
  2: [0, 0, 0, 1, 1, 0, 1]
  3: [0, 0, 1, 0, 1, 1, 0]
  4: [0, 1, 1, 0, 0, 1, 1]
  5: [0, 1, 0, 0, 0, 0, 1]

Solving with trace:
Choose column A (size=2)
Try row 0, also covering [D, G]
  Choose column B (size=0)
  Dead end — column B has no rows, backtracking
Try row 1, also covering [D]
  Choose column E (size=1)
  Try row 3, also covering [F, C]
    Choose column B (size=1)
    Try row 5, also covering [G]
      ** All columns covered — solution found: rows [1, 3, 5]
    Uncovered column B, backtracking
  Uncovered column E, backtracking
Uncovered column A, backtracking

Solution rows: [1, 3, 5]
Verification:
  Row 1: covers columns ['A', 'D']
  Row 3: covers columns ['C', 'E', 'F']
  Row 5: covers columns ['B', 'G']


## Sudoku Solver

Sudoku is a natural fit for exact cover. A completed Sudoku board satisfies four types of constraints, each with 81 conditions:

1. **Cell** — each of the 81 cells contains exactly one digit.
2. **Row-digit** — each row contains each digit 1-9 exactly once.
3. **Column-digit** — each column contains each digit 1-9 exactly once.
4. **Box-digit** — each 3x3 box contains each digit 1-9 exactly once.

This gives a matrix with **324 columns** (4 x 81). Each possible placement (row, column, digit) becomes a row in the matrix with exactly 4 ones — one for each constraint it satisfies. Given digits are handled by only generating that single row for the cell, rather than all 9 candidates.

In [7]:
def sudoku_to_exact_cover(puzzle):
    """Convert a 9x9 Sudoku grid to an exact cover problem.

    puzzle: list of 81 ints (0 = empty, 1-9 = given digit)
    Returns (matrix, row_info) where row_info[i] = (row, col, digit).
    """
    rows = []
    row_info = []
    for r in range(9):
        for c in range(9):
            val = puzzle[r * 9 + c]
            digits = [val] if val != 0 else range(1, 10)
            for d in digits:
                b = (r // 3) * 3 + (c // 3)
                row = [0] * 324
                row[r * 9 + c] = 1              # cell constraint
                row[81 + r * 9 + (d - 1)] = 1   # row-digit constraint
                row[162 + c * 9 + (d - 1)] = 1  # column-digit constraint
                row[243 + b * 9 + (d - 1)] = 1  # box-digit constraint
                rows.append(row)
                row_info.append((r, c, d))
    return rows, row_info


def solve_sudoku(puzzle):
    """Solve a Sudoku puzzle using DLX. Returns the solved grid."""
    matrix, row_info = sudoku_to_exact_cover(puzzle)
    dlx = DLX(matrix)
    for solution in dlx.solve():
        grid = [0] * 81
        for row_idx in solution:
            r, c, d = row_info[row_idx]
            grid[r * 9 + c] = d
        return grid
    return None


def print_sudoku(grid):
    for r in range(9):
        if r > 0 and r % 3 == 0:
            print("------+-------+------")
        row_str = ""
        for c in range(9):
            if c > 0 and c % 3 == 0:
                row_str += " | "
            val = grid[r * 9 + c]
            row_str += f" {val if val != 0 else '.'}"
        print(row_str)


# A well-known example puzzle.
puzzle = [
    5, 3, 0, 0, 7, 0, 0, 0, 0,
    6, 0, 0, 1, 9, 5, 0, 0, 0,
    0, 9, 8, 0, 0, 0, 0, 6, 0,
    8, 0, 0, 0, 6, 0, 0, 0, 3,
    4, 0, 0, 8, 0, 3, 0, 0, 1,
    7, 0, 0, 0, 2, 0, 0, 0, 6,
    0, 6, 0, 0, 0, 0, 2, 8, 0,
    0, 0, 0, 4, 1, 9, 0, 0, 5,
    0, 0, 0, 0, 8, 0, 0, 7, 9,
]

print("Puzzle:")
print_sudoku(puzzle)

solution = solve_sudoku(puzzle)
print("\nSolved:")
print_sudoku(solution)

Puzzle:
 5 3 . |  . 7 . |  . . .
 6 . . |  1 9 5 |  . . .
 . 9 8 |  . . . |  . 6 .
------+-------+------
 8 . . |  . 6 . |  . . 3
 4 . . |  8 . 3 |  . . 1
 7 . . |  . 2 . |  . . 6
------+-------+------
 . 6 . |  . . . |  2 8 .
 . . . |  4 1 9 |  . . 5
 . . . |  . 8 . |  . 7 9

Solved:
 5 3 4 |  6 7 8 |  9 1 2
 6 7 2 |  1 9 5 |  3 4 8
 1 9 8 |  3 4 2 |  5 6 7
------+-------+------
 8 5 9 |  7 6 1 |  4 2 3
 4 2 6 |  8 5 3 |  7 9 1
 7 1 3 |  9 2 4 |  8 5 6
------+-------+------
 9 6 1 |  5 3 7 |  2 8 4
 2 8 7 |  4 1 9 |  6 3 5
 3 4 5 |  2 8 6 |  1 7 9


In [8]:
# A harder puzzle with fewer givens.
hard_puzzle = [
    0, 0, 0, 6, 0, 0, 4, 0, 0,
    7, 0, 0, 0, 0, 3, 6, 0, 0,
    0, 0, 0, 0, 9, 1, 0, 8, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 5, 0, 1, 8, 0, 0, 0, 3,
    0, 0, 0, 3, 0, 6, 0, 4, 5,
    0, 4, 0, 2, 0, 0, 0, 6, 0,
    9, 0, 3, 0, 0, 0, 0, 0, 0,
    0, 2, 0, 0, 0, 0, 1, 0, 0,
]

print("Puzzle:")
print_sudoku(hard_puzzle)

hard_solution = solve_sudoku(hard_puzzle)
print("\nSolved:")
print_sudoku(hard_solution)

Puzzle:
 . . . |  6 . . |  4 . .
 7 . . |  . . 3 |  6 . .
 . . . |  . 9 1 |  . 8 .
------+-------+------
 . . . |  . . . |  . . .
 . 5 . |  1 8 . |  . . 3
 . . . |  3 . 6 |  . 4 5
------+-------+------
 . 4 . |  2 . . |  . 6 .
 9 . 3 |  . . . |  . . .
 . 2 . |  . . . |  1 . .

Solved:
 5 8 1 |  6 7 2 |  4 3 9
 7 9 2 |  8 4 3 |  6 5 1
 3 6 4 |  5 9 1 |  7 8 2
------+-------+------
 4 3 8 |  9 5 7 |  2 1 6
 2 5 6 |  1 8 4 |  9 7 3
 1 7 9 |  3 2 6 |  8 4 5
------+-------+------
 8 4 5 |  2 1 9 |  3 6 7
 9 1 3 |  7 6 8 |  5 2 4
 6 2 7 |  4 3 5 |  1 9 8


## How It Works — Recap

1. **Model the problem as exact cover.** Build a binary matrix where each row is a candidate choice and each column is a constraint. A solution is a subset of rows that covers every column exactly once.
2. **Build the sparse linked structure.** Only 1-entries become `Node` objects, linked in four directions. Column headers form a horizontal list off a master header.
3. **Choose the column with the fewest nodes** (S-heuristic). This minimizes the branching factor.
4. **Try each row in that column.** For each candidate row, include it in the partial solution.
5. **Cover** every column touched by the chosen row — unlink the column header and remove all conflicting rows from the structure. This is where the pointers "dance" out.
6. **Recurse.** If the header's horizontal list is empty, all constraints are satisfied — record the solution.
7. **Backtrack.** Uncover columns in reverse order — the pointers dance back into place — and try the next row.
8. **Repeat** until all possibilities have been explored.

The beauty of DLX is that covering and uncovering are symmetric, constant-time operations per node. No auxiliary data structures are needed — the linked list *is* the undo stack.

## References

- Knuth, D.E. (2000). **"Dancing Links."** *Millennial Perspectives in Computer Science*, 159-187. [arXiv:cs/0011047](https://arxiv.org/abs/cs/0011047)
- [Wikipedia: Dancing Links](https://en.wikipedia.org/wiki/Dancing_links)
- [Wikipedia: Knuth's Algorithm X](https://en.wikipedia.org/wiki/Knuth%27s_Algorithm_X)